# Mini project: 2D Convolution on the Raspberry Pi
## Python → C → ARM64 Assembly

In this mini project you run the **same operation**, a 2D convolution, written in three languages on
a Raspberry Pi 5, and then step by step in assembly. You will measure how long each version
takes and find out *why* the lower levels are faster.

You will not write a program from scratch. The code is ready and commented. Your job is to
**take a photo with the Raspberry Pi camera, run the code on it, read it, write three
pieces of it yourself (one in C, two in assembly), and explain what you see** with your own
numbers.

The camera must be connected **before** the Raspberry Pi is powered on.

### How to use this notebook

- Run a cell with **Shift + Enter**. Run the cells **in order**, from top to bottom.
- A cell marked **✏️** is a question: double-click it, write your answer below
  **Your answer:**, then Shift + Enter.
- Save often (**Ctrl + S**).

### What you hand in

This notebook, **exported as HTML** with all outputs and answers (see the last section).

---
## Part 0. Check the setup

The cell below checks that the system is 64-bit ARM (`aarch64`), that the camera is
detected, builds the C and assembly code with `make`, and runs a correctness test: every
version must give exactly the same result as Python on 13 image sizes.

**Expected:** `aarch64`, a camera in the list (for example `0 : ov5647 [2592x1944 ...]`),
then the build, then a line ending in `all OK`. The two versions marked **(yours)** are the
ones you write later: `C v2 rows (yours)` and `ASM v1 brute (yours)` say `MISMATCH` until
you write them in Parts 3 and 4.

If it says `No cameras available!`: shut the Pi down, unplug it, check the camera cable at
both ends, and power it on again. The camera is only detected if it is connected **before**
the Pi is powered on.

In [ ]:
!uname -m
!cat /proc/device-tree/model; echo
!rpicam-hello --list-cameras 2>/dev/null
!make
!python3 conv_bench.py --test

---
## Part 1. What is a convolution?

A **kernel** is a small grid of numbers, here 3 × 3. It slides over the image. At each
position, every pixel under the kernel is multiplied by the kernel number on top of it, and
the 9 products are added. That sum is one pixel of the output:

$$
\text{out}[y][x] = \sum_{k_y=0}^{2}\ \sum_{k_x=0}^{2}\ \text{in}[y+k_y][x+k_x] \cdot k[k_y][k_x]
$$

The kernel never hangs outside the image, so the output is 2 pixels smaller in each
direction. Image filters (blur, edge detection) and the neural networks that recognize
images are built on exactly this operation.

Run the cell below. It shows a small 6 × 8 image, the kernel, and how output pixel
**(0, 0)** is computed: the yellow 3 × 3 window is multiplied by the kernel and summed.

In [ ]:
from conv_demo import notebook_image, notebook_race, notebook_step, take_photo

notebook_step(0, 0)

✏️ **Q1.** Compute output pixel **out[1][2] by hand** for this new image and kernel. Write the
9 products and the result. *(Rows and columns are numbered from 0. The window for
out[1][2] starts at row 1, column 2 of the input.)*

**Input image** (5 × 6)

| | col 0 | col 1 | col 2 | col 3 | col 4 | col 5 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| **row 0** | 8 | 6 | 2 | 2 | 6 | 6 |
| **row 1** | 5 | 8 | 9 | 6 | 5 | 2 |
| **row 2** | 4 | 7 | 7 | 9 | 5 | 3 |
| **row 3** | 0 | 7 | 3 | 1 | 7 | 9 |
| **row 4** | 4 | 8 | 9 | 6 | 0 | 1 |

**Kernel** (3 × 3)

| | col 0 | col 1 | col 2 |
| --- | ---: | ---: | ---: |
| **row 0** | 0 | −2 | 1 |
| **row 1** | 1 | 1 | −1 |
| **row 2** | −1 | −2 | 2 |

**Your answer:** *(double-click this cell and write it here)*

### The image of this project: your photo

Every part of this project processes **a photo you take now with the Raspberry Pi camera**.
To a computer a grayscale photo is just a grid of numbers, one per pixel, from 0 (black) to
255 (white). Whatever the photo shows, every version does exactly the same amount of work,
so everybody's times are comparable.

The project uses the **edge-detection kernel**

```
-1 -1 -1
-1  8 -1
-1 -1 -1
```

multiplies the centre pixel by 8 and subtracts its 8 neighbours. Where the brightness does
not change (a wall, the sky) the result is 0; where it changes (the outline of a face, a
letter), the result is large.

Run the cell below. A camera preview opens for 3 seconds, then the Raspberry Pi takes a
grayscale photo of **2592 × 1944 pixels** and saves it as `photo.pgm`. It does not have to
be your face: point the camera at anything with clear shapes (a hand, a book, a drawing).
The photo will be part of the notebook you hand in.

If the cell says **No camera detected**, follow the steps it gives and run it again. To
retake the photo, just run the cell again: the new photo replaces the old one.

In [ ]:
take_photo()
notebook_image()

✏️ **Q2.** Your photo is 2592 × 1944 pixels. How many output pixels does the convolution
produce, and how many **multiply-adds** does it need for the whole photo? Show the
calculation.

**Your answer:** *(double-click this cell and write it here)*

✏️ **Q3.** In the output, which parts of your photo turned dark (strong edges) and which
stayed white? Use the kernel to explain why a flat area gives 0. *(Hint: add up the 9
numbers of the kernel.)*

**Your answer:** *(double-click this cell and write it here)*

---
## Part 2. The same loops in three languages

### Python

Four nested loops: every output pixel, then every kernel position.

```python
for y in range(oh):                  # every output row ...
    for x in range(ow):              # ... every output column
        acc = 0
        for ky in range(3):          # every kernel row
            for kx in range(3):      # every kernel column
                acc += img[y + ky][x + kx] * k[ky][kx]
        out[y][x] = acc
```

Python does not run this directly on the CPU. The **interpreter** reads each operation
while the program runs: it looks up `img`, finds row `y + ky` in a list, finds the element,
checks that both values are numbers, creates a new integer object for the product, and so
on. Each `+` and `*` costs many steps.

### C v1 (`src/conv.c`): column order

The same four loops. The difference is *when* the work happens: the compiler (`gcc -O2`)
translates them to machine instructions **once**, before the program runs.

This first C version walks the image **column by column**: the outer loop is `x`. Next to
the loops and the operations, the comments show the RISC-V instructions you would write in
Ripes, with `a0` = `in`, `a1` = `out`, `a2` = `k`, `a4` = `w`, `s0` = `x`, `s1` = `y`,
`s2` = `ky`, `s3` = `kx`, `s4` = `acc`, `s5` = `oh`, `s6` = `ow`. Notice how **one line of
C** becomes **many instructions**:

```c
void conv_c_cols(int *in, int *out, int *k, int h, int w)
{
    int oh = h - 2;                               // output height
    int ow = w - 2;                               // output width

    for (int x = 0; x < ow; x++) {                //     li   s0, 0           # x = 0
                                                  // loop_x:
        for (int y = 0; y < oh; y++) {            //     li   s1, 0           # y = 0
                                                  // loop_y:
            int acc = 0;                          //     li   s4, 0           # acc = 0
            for (int ky = 0; ky < 3; ky++) {      //     li   s2, 0           # ky = 0
                                                  // loop_ky:
                for (int kx = 0; kx < 3; kx++) {  //     li   s3, 0           # kx = 0
                                                  // loop_kx:
                    acc += in[(y + ky) * w + (x + kx)] * k[ky * 3 + kx];
                                                  //     add  t0, s1, s2      # y + ky
                                                  //     mul  t0, t0, a4      # (y + ky) * w
                                                  //     add  t1, s0, s3      # x + kx
                                                  //     add  t0, t0, t1      # index into in[]
                                                  //     slli t0, t0, 2       # index * 4 bytes
                                                  //     add  t0, a0, t0      # address of the pixel
                                                  //     lw   t2, 0(t0)       # pixel
                                                  //     li   t3, 3
                                                  //     mul  t3, s2, t3      # ky * 3
                                                  //     add  t3, t3, s3      # index into k[]
                                                  //     slli t3, t3, 2       # index * 4 bytes
                                                  //     add  t3, a2, t3      # address of the coefficient
                                                  //     lw   t4, 0(t3)       # coef
                                                  //     mul  t2, t2, t4      # pixel * coef
                                                  //     add  s4, s4, t2      # acc += pixel * coef
                }                                 //     addi s3, s3, 1       # kx++
                                                  //     li   t5, 3
                                                  //     blt  s3, t5, loop_kx # next kx while kx < 3
            }                                     //     addi s2, s2, 1       # ky++
                                                  //     blt  s2, t5, loop_ky # next ky while ky < 3
            out[y * ow + x] = acc;                //     mul  t0, s1, s6      # y * ow
                                                  //     add  t0, t0, s0      # + x
                                                  //     slli t0, t0, 2       # * 4 bytes
                                                  //     add  t0, a1, t0      # address of out[y * ow + x]
                                                  //     sw   s4, 0(t0)       # store acc
        }                                         //     addi s1, s1, 1       # y++
                                                  //     blt  s1, s5, loop_y  # next y while y < oh
    }                                             //     addi s0, s0, 1       # x++
                                                  //     blt  s0, s6, loop_x  # next x while x < ow
}
```

### How the image sits in memory

- The image is one flat block of memory, **row after row**, so pixel `[r][c]` is at index
  `r * w + c`. Neighbours in a row are neighbours in memory; neighbours in a column are a
  whole row apart (2592 × 4 bytes = 10,368 bytes).
- The CPU never reads one pixel from RAM. It brings a whole **cache line** of 64 bytes
  (16 pixels) into its fast **cache** memory, expecting the next pixels to be used soon.

C v1 walks down the columns: every step jumps 10,368 bytes and needs a different cache
line, and the cache has to keep thousands of lines to reuse them for the next column. In
Part 3 you will write **C v2**, which walks the image in the order it is stored. The
compiler does **not** fix the order for you: it is the programmer's choice.

### How Python calls C and assembly

`make` turns `src/conv.c` and `src/conv_asm.S` into **shared libraries** (`build/*.so`).
`conv_bench.py` loads them with Python's `ctypes` module and calls the functions as if they
were Python functions, passing the image as a block of memory. Python only measures the
time and checks the output; the work runs as native machine code.

### From RISC-V (Ripes) to ARM64

In Ripes you wrote **RISC-V** assembly. The Raspberry Pi's CPU speaks **ARM64**
(also called AArch64). Both are RISC architectures built on the same ideas: registers,
instructions with a destination and two operands, and memory reached only through loads
and stores. What changes is mostly how things are written. Use these two tables as a
dictionary while you read the ARM64 code.

**Registers**

| Use | RISC-V (Ripes) | ARM64 |
| --- | --- | --- |
| Function arguments and result | `a0`–`a7` | `x0`–`x7` |
| Temporaries (free to overwrite) | `t0`–`t6` | `x9`–`x15` (also `x16`, `x17`) |
| Saved (must be restored before returning) | `s0`–`s11` | `x19`–`x28` |
| Return address | `ra` | `x30` (also called `lr`) |
| Stack pointer | `sp` | `sp` |
| Always zero | `zero` | `xzr` / `wzr` |
| Size | 32 bits (RV32) | `xN` = 64 bits; `wN` = the low 32 bits of `xN` |

In this project, pixels are 32-bit numbers, so pixel arithmetic uses `w` registers; memory
addresses and loop counters use `x` registers.

**Instructions**

| What it does | RISC-V (Ripes) | ARM64 | How to read the ARM64 |
| --- | --- | --- | --- |
| Load a constant | <code style="white-space:nowrap">li t0, 3</code> | <code style="white-space:nowrap">mov x15, #3</code> | `x15` = 3. Constants start with `#`. |
| Copy a register | <code style="white-space:nowrap">mv s4, s2</code> | <code style="white-space:nowrap">mov w4, w2</code> | `w4` = `w2`. |
| Add / add a constant | <code style="white-space:nowrap">add t0, t1, t2</code> / <code style="white-space:nowrap">addi t0, t0, 1</code> | <code style="white-space:nowrap">add x0, x1, x2</code> / <code style="white-space:nowrap">add x0, x0, #1</code> | `x0` = `x1` + `x2` / `x0` = `x0` + 1. The same `add` does both. |
| Multiply | <code style="white-space:nowrap">mul t0, t1, t2</code> | <code style="white-space:nowrap">mul w0, w1, w2</code> | `w0` = `w1` × `w2`. |
| Shift left | <code style="white-space:nowrap">slli t0, t0, 2</code> | <code style="white-space:nowrap">lsl x0, x0, #2</code> | `x0` = `x0` << 2, that is `x0` × 4. |
| Load a word from memory | <code style="white-space:nowrap">lw t0, 4(a0)</code> | <code style="white-space:nowrap">ldr w0, [x0, #4]</code> | `w0` = the 4-byte word at address `x0` + 4. Inside `[ ]`: the base address, then the offset in bytes. |
| Store a word | <code style="white-space:nowrap">sw t0, 0(a1)</code> | <code style="white-space:nowrap">str w0, [x1]</code> | Write `w0` (4 bytes) at address `x1`. With no offset, it is `x1` itself. |
| Load, then advance the pointer | <code style="white-space:nowrap">lw t0, 0(a0)</code> + <code style="white-space:nowrap">addi a0, a0, 4</code> | <code style="white-space:nowrap">ldr w0, [x0], #4</code> | `w0` = the word at address `x0`, **then** `x0` = `x0` + 4. The `#4` after `]` is added to the pointer after loading (post-increment), ready for the next word. |
| Load `array[index]` (4-byte words) | <code style="white-space:nowrap">slli</code> + <code style="white-space:nowrap">add</code> + <code style="white-space:nowrap">lw</code> | <code style="white-space:nowrap">ldr w0, [x1, x2, lsl #2]</code> | `w0` = the word at address `x1` + `x2` × 4. `x1` is the address of the array, `x2` the index, and `lsl #2` multiplies the index by 4 because each word is 4 bytes. |
| Branch if less than | <code style="white-space:nowrap">blt t0, t1, loop</code> | <code style="white-space:nowrap">cmp x0, x1</code> then <code style="white-space:nowrap">b.lt loop</code> | `cmp` compares `x0` with `x1` and remembers the result (in the *flags*); `b.lt` jumps to `loop` if it was "less than". |
| Branch if (not) zero | <code style="white-space:nowrap">beqz t0, done</code> / <code style="white-space:nowrap">bnez t0, loop</code> | <code style="white-space:nowrap">cbz x0, done</code> / <code style="white-space:nowrap">cbnz x0, loop</code> | Jump if `x0` is zero / is not zero. |
| Count down and loop | <code style="white-space:nowrap">addi t0, t0, -1</code> + <code style="white-space:nowrap">bnez t0, loop</code> | <code style="white-space:nowrap">subs x0, x0, #1</code> then <code style="white-space:nowrap">b.ne loop</code> | `x0` = `x0` − 1 and set the flags (the `s` in `subs`); `b.ne` jumps if the result was not zero. |
| Jump / return | <code style="white-space:nowrap">j loop</code> / <code style="white-space:nowrap">ret</code> | <code style="white-space:nowrap">b loop</code> / <code style="white-space:nowrap">ret</code> | Always jump to `loop` / return from the function. |
| Multiply and add (MAC) | <code style="white-space:nowrap">mul</code> + <code style="white-space:nowrap">add</code> (RV32IM has no MAC) | <code style="white-space:nowrap">madd w0, w1, w2, w0</code> | `w0` = `w0` + `w1` × `w2`. The 4th register is the one added. |
| Comment | `# ...` | `// ...` | |

Three things to remember: in ARM64 constants start with `#`, memory operands go inside
`[ ]`, and comparisons are separate from branches (`cmp` sets flags, `b.lt` / `b.ne` read
them).

---
## Part 3. Hands-on 1: write C v2 (row order)

Write the convolution in C yourself, this time walking the image **row by row**, the order
it is stored in memory. The cell below has the function with its signature; write its body
and run the cell. It compiles your code and runs the benchmark without Python (a few
seconds); your row is **C v2 rows (yours)** and the reference is C v1.

**Hints**

- `out[y][x]` is the sum of the 9 products `in[y + ky][x + kx] * k[ky][kx]`, for `ky` and
  `kx` from 0 to 2. It is the same work as C v1: only the order of the loops changes.
- Everything is stored row by row in one flat block: `in[r][c]` is `in[r * w + c]`,
  `out[y][x]` is `out[y * ow + x]`, and `k[ky][kx]` is `k[ky * 3 + kx]`.
- To walk the image row by row, the **outer** loop goes over the rows (`y`, from 0 to
  `oh - 1`) and the next one over the columns (`x`, from 0 to `ow - 1`).
- Start each output pixel with `int acc = 0;` and store it in `out` when its 9 products
  are added.

**Your row must say `OK`**, and so must the *extra check with random images and kernels*
printed below the table. (The photo's edge kernel is symmetric, so a few mistakes, such as
swapping `ky` and `kx` in the kernel index, only show up with other kernels.)

- `NOT BUILT`: the compiler found an error. The message shows the line of your code and
  what is wrong. Fix it and run the cell again.
- `MISMATCH`: it builds but computes the wrong result. Check the loop limits and the
  indexes.
- *Your code crashed the program*: it read or wrote outside the image. Check the loop
  limits.

In [ ]:
from conv_demo import build_c_v2

my_c_v2 = """
void conv_c_rows(int *in, int *out, int *k, int h, int w)
{
    int oh = h - 2;    // output height
    int ow = w - 2;    // output width

    // Write the convolution here: the same work as C v1 in src/conv.c,
    // walking the image ROW BY ROW. See the hints in the notebook.

}
"""

build_c_v2(my_c_v2)

---
## Part 4. Hands-on 2: write the core of assembly v1

Now translate your C v2 into ARM64 assembly. **ASM v1** is a direct, "brute force"
translation: the same four loops, and every array index computed from scratch with `mul`,
exactly as the C source says. Almost all of it is written for you: the size check, the four
loops and the pixel load. You write **the core of the work**: load the kernel coefficient,
multiply and accumulate, and store the output pixel.

### Registers

The same roles as the RISC-V comments of C v1 (Part 2):

| Value | RISC-V (Ripes) | ARM64 |
| --- | --- | --- |
| `in`, `out`, `k`, `h`, `w` (arguments) | `a0`–`a4` | `x0`–`x4` |
| `oh`, `ow` | `s5`, `s6` | `x5`, `x6` |
| `x`, `y` | `s0`, `s1` | `x7`, `x8` |
| `ky`, `kx` | `s2`, `s3` | `x10`, `x11` |
| `acc` | `s4` | `w9` |
| pixel, coefficient | `t2`, `t4` | `w14`, `w16` |
| temporaries | `t0`, `t1`, `t3` | `x12`, `x13`, `x15` |

### The code that is written

Read it line by line. As in Ripes, the loops start at **labels** with names; each loop ends
with `cmp` (compare the counter with its limit) and `b.lt label` (if less than, jump back to
the label).

| Label | Start of | Jumped back to while |
| --- | --- | --- |
| `loop_y` | an output row (`x = 0`) | `y < oh` |
| `loop_x` | an output pixel (`acc = 0`, `ky = 0`) | `x < ow` |
| `loop_ky` | a kernel row (`kx = 0`) | `ky < 3` |
| `loop_kx` | a kernel column (load the pixel) | `kx < 3` |

Before your code (size check, start of the loops, pixel load):

```asm
conv_asm_v1:
        cmp     x3, #3                  // image smaller than the kernel?
        b.lt    done                    //   then there is nothing to do
        cmp     x4, #3
        b.lt    done
        sub     x5, x3, #2              // oh = h - 2
        sub     x6, x4, #2              // ow = w - 2

        mov     x8, #0                  // y = 0
loop_y:
        mov     x7, #0                  //   x = 0
loop_x:
        mov     w9, #0                  //     acc = 0
        mov     x10, #0                 //     ky = 0
loop_ky:
        mov     x11, #0                 //       kx = 0
loop_kx:
        add     x12, x8, x10            //         y + ky
        mul     x12, x12, x4            //         (y + ky) * w
        add     x13, x7, x11            //         x + kx
        add     x12, x12, x13           //         index into in[]
        ldr     w14, [x0, x12, lsl #2]  //         pixel = in[index]   (lsl #2: * 4 bytes)
```

After your code (the end of the `x` and `y` loops):

```asm
        add     x7, x7, #1              //     x++
        cmp     x7, x6
        b.lt    loop_x                  //     next x while x < ow
        add     x8, x8, #1              //   y++
        cmp     x8, x5
        b.lt    loop_y                  //   next y while y < oh
done:
        ret
```

### Your part

In the cell below, write the 3 steps marked in the comments. The lines between step 2 and
step 3 (the end of the `kx` and `ky` loops) are given: do not change them. Hints:

- **Step 1** is like loading the pixel: compute the index `ky * 3 + kx` in `x15`
  (`mov x15, #3` gives you the 3; then `mul` and `add`), and load with
  `ldr w16, [x2, x15, lsl #2]`. Four instructions.
- **Step 2** is one `mul` (pixel × coef, into `w14`) and one `add` (into `acc`, `w9`). The
  `madd` that does both comes later, in Part 7.
- **Step 3** computes the index `y * ow + x` in `x12` (one `mul`, one `add`) and stores with
  `str w9, [x1, x12, lsl #2]`. Three instructions.
- Pixels are 32-bit numbers: use `w` registers for them (`w9`, `w14`, `w16`) and `x`
  registers for indexes and addresses.
- The RISC-V comments of C v1 in Part 2 and the translation tables tell you the ARM64
  instruction for each RISC-V one.

**Your row is `ASM v1 brute (yours)` and must say `OK`**, and so must the extra check with
random images and kernels below the table. If it says `NOT BUILT`, the message
shows the line of your code with the error. If the cell says your code *crashed*, an index
is wrong.

In [ ]:
from conv_demo import build_v1

my_v1 = """
        // Step 1. Load the coefficient k[ky * 3 + kx] into w16.
        //    RISC-V: li t3, 3 / mul t3, s2, t3 / add t3, t3, s3 / slli + add + lw

        // Step 2. acc += pixel * coef   (acc is w9, pixel is w14, coef is w16)

        // ---- Given: the two kernel loops end here (do not change) ----
        add     x11, x11, #1            //         kx++
        cmp     x11, #3
        b.lt    loop_kx                 //         next kx while kx < 3
        add     x10, x10, #1            //       ky++
        cmp     x10, #3
        b.lt    loop_ky                 //       next ky while ky < 3

        // Step 3. out[y * ow + x] = acc
        //    RISC-V: mul t0, s1, s6 / add t0, t0, s0 / slli + add + sw
"""

build_v1(my_v1)

✏️ **Q4.** In your ASM v1, count the instructions that run for **one** multiply-add (from
the label `loop_kx` to the `b.lt loop_kx` that repeats it). How many of them do the useful work (load the
pixel, load the coefficient, multiply, add to `acc`), and how many only compute indexes or
control the loop? The same loop in RISC-V (the C v1 comments) needs 18 instructions: which
ARM64 instructions do the job of several RISC-V ones?

**Your answer:** *(double-click this cell and write it here)*

---
## Part 5. Measure all the versions

The cell below runs every version on your photo, with the edge-detection kernel. Python
runs once (**about 9 seconds**); each C and assembly version runs 5 times and the median is
shown. Every output is compared with Python's: `OK` means identical.

Close other programs before running it.

In [ ]:
!python3 conv_bench.py

✏️ **Q5.** Copy your numbers into the table (two decimals).

| Version | Time (ms) | vs Python |
| --- | ---: | ---: |
| Python | | 1.0x |
| C v1 column order | | |
| C v2 rows (yours) | | |
| ASM v1 brute (yours) | | |
| ASM v2 pointers | | |
| ASM v3 MAC (yours) | | |
| ASM v4 NEON | | |

*(Double-click to fill it in.)*

✏️ **Q6.** Where was the **biggest** jump in speed, and why?
Is a direct translation to assembly (ASM v1) automatically faster than C v2? Look at
your numbers.

**Your answer:** *(double-click this cell and write it here)*

✏️ **Q7.** C v1 and your C v2 are the same language, the same compiler and the same
operations; only the order of the loops is different. How many times faster is your C v2
than C v1 on your Pi? Explain the difference with the words *memory*, *row* and *cache
line*.

**Your answer:** *(double-click this cell and write it here)*

---
## Part 6. Climbing the assembly ladder

Each assembly version changes **one idea** compared to the previous one, so each speed-up
has one cause. Open `src/conv_asm.S` to read the full code; every line is commented.

### v1 → v2: stop computing indexes inside the loop

v1 recomputes `(y + ky) * w + (x + kx)` with multiplications for every single multiply-add,
and reloads the kernel from memory every time. v2 instead:

- loads the 9 kernel numbers **once** into registers `w9`–`w17`;
- keeps **three pointers** (`x7`, `x8`, `x2`) on the three image rows under the kernel, and
  moves them 4 bytes to the right after each output pixel. No index arithmetic at all;
- writes out the 9 kernel positions one after another (no `ky`, `kx` loops).

`ldr w20, [x7], #4` loads a pixel **and** moves the pointer to the next one, in a single
instruction (*post-increment*). One output pixel of v2:

```asm
2:      ldr     w20, [x7], #4           // row y:   in[y][x], then p0 += 4
        mul     w19, w20, w9            //   acc  = pixel * k[0]
        ldr     w20, [x7]               //          in[y][x+1]
        mul     w20, w20, w10           //   product = pixel * k[1]
        add     w19, w19, w20           //   acc += product
        ...                             //   (the same for the other 7 positions)
        str     w19, [x1], #4           // *out++ = acc
        subs    x6, x6, #1              // columns left--
        b.ne    2b
```

### v2 → v3: the MAC instruction (your turn, Part 7)

In v2 every kernel position after the first needs `mul` and then `add`, exactly as you
would write it in Ripes: RISC-V (RV32IM) has no instruction that does both. ARM64 does: a
**multiply-accumulate** (MAC) instruction:

```asm
madd    Wd, Wn, Wm, Wa      // Wd = Wa + Wn * Wm
```

### v3 → v4: NEON, 4 pixels at once

The Raspberry Pi's CPU has **NEON** vector registers (`v0`–`v31`) of 128 bits, which hold
4 numbers of 32 bits each. One vector instruction works on all 4 at the same time:

```asm
ldr     q16, [x7]                   // load 4 pixels: in[y][x .. x+3]
mla     v20.4s, v16.4s, v0.s[1]     // 4 accumulators += 4 pixels * k[1]
```

v4 computes **4 output pixels per loop iteration**. The C compiler did not use these
instructions for `conv.c`; writing assembly by hand let us use them.

✏️ **Q8.** Count the instructions that run for **one output pixel** in v2 (loads,
multiplies, adds, the store, and the 2 loop instructions). How many is that per
multiply-add? Compare with your answer to Q4 for assembly v1.

**Your answer:** *(double-click this cell and write it here)*

---
## Part 7. Hands-on 3: write assembly v3

Now you change real ARM64 code. You only touch **one output pixel**: the 9 loads and 9
multiply-adds that v2 runs for each pixel. That code is in the cell below; you edit it
there and run the cell. The rest of the function (loops, pointers, saving registers) is
already written in `src/conv_student.S`.

### The idea: fuse `mul` + `add` into `madd`

In v2, every kernel position after the first takes two instructions, exactly as you would
write it in Ripes:

```
ARM64 v2 (two instructions)                     RISC-V in Ripes (also two)
mul   w20, w20, w11   // product = pixel*k[2]   mul  t2, t2, t4     # pixel * coef
add   w19, w19, w20   // acc = acc + product    add  s4, s4, t2     # acc += pixel * coef
```

ARM64 can do both in **one** instruction, `madd` (multiply-add):

```
madd  Wd, Wn, Wm, Wa      // Wd = Wa + Wn * Wm
```

### Worked example: the first pair is already done

The first pair, `mul w20, w20, w10` + `add w19, w19, w20`, is already replaced in the cell:

```
madd  w19, w20, w10, w19  // acc = acc + pixel * k[1]
```

| Operand | Register | Meaning |
| --- | --- | --- |
| `Wd` (result) | `w19` | the accumulator `acc`: the sum goes back into it |
| `Wn` (multiply) | `w20` | the pixel just loaded |
| `Wm` (multiply) | `w10` | the kernel number `k[1]` |
| `Wa` (add) | `w19` | the old value of `acc` |

### Your task

In the cell below, replace each of the **7 pairs marked `TODO`** with one `madd`, following
the example. Only the kernel register changes from pair to pair (`w11`, `w12`, … `w17`).
Then run the cell (Shift + Enter). It builds your code and runs the benchmark without
Python (a few seconds); the reference is C v1.

- **Your row must say `OK`**, the extra check below the table must say OK too, and the
  message must say **8 madd instructions**.
- `NOT BUILT`: the assembler found an error. The message shows the line of your code and
  what is wrong. Fix it and run the cell again.
- `MISMATCH`: it builds but computes the wrong result. Check the order of the operands.
- Made a mess? **Ctrl + Z** in the cell undoes your edits. The original pairs are also in
  v2, in `src/conv_asm.S`.

In [ ]:
from conv_demo import build_v3

my_pixel = """
        ldr     w20, [x7], #4           // row y:   in[y][x], then p0 += 4
        mul     w19, w20, w9            //   acc  = pixel * k[0]
        ldr     w20, [x7]               //          in[y][x+1]
        madd    w19, w20, w10, w19      // EXAMPLE (was TODO 1): acc = acc + pixel * k[1]
        ldr     w20, [x7, #4]           //          in[y][x+2]
        mul     w20, w20, w11           // TODO 2: fuse this MUL ...
        add     w19, w19, w20           //         ... and this ADD into one MADD

        ldr     w20, [x8], #4           // row y+1
        mul     w20, w20, w12           // TODO 3: fuse this MUL ...
        add     w19, w19, w20           //         ... and this ADD into one MADD
        ldr     w20, [x8]
        mul     w20, w20, w13           // TODO 4: fuse this MUL ...
        add     w19, w19, w20           //         ... and this ADD into one MADD
        ldr     w20, [x8, #4]
        mul     w20, w20, w14           // TODO 5: fuse this MUL ...
        add     w19, w19, w20           //         ... and this ADD into one MADD

        ldr     w20, [x2], #4           // row y+2
        mul     w20, w20, w15           // TODO 6: fuse this MUL ...
        add     w19, w19, w20           //         ... and this ADD into one MADD
        ldr     w20, [x2]
        mul     w20, w20, w16           // TODO 7: fuse this MUL ...
        add     w19, w19, w20           //         ... and this ADD into one MADD
        ldr     w20, [x2, #4]
        mul     w20, w20, w17           // TODO 8: fuse this MUL ...
        add     w19, w19, w20           //         ... and this ADD into one MADD
"""

build_v3(my_pixel)

✏️ **Q9.** Compare the time of your v3 with v2. How many instructions per output pixel
does v3 run (use your count from Q8)? Did the speed change by the same proportion as the
instruction count? Propose one reason why it might not.

**Your answer:** *(double-click this cell and write it here)*

---
## Part 8. The race

Now see it live, on your photo. Python processes it on a **CPU core of its own**. Meanwhile,
on another core, the six C and assembly versions process the same photo **one after
another**, never at the same time, so no version slows another one down.

- **Panel 1 (one photo):** the progress of the photo for every version. The C and assembly
  bars fill one after another in less than a second, each with its time, while Python's
  bar crawls for about 9 seconds.
- **Panel 2:** how many photos each C and assembly version **could** process in the time
  Python has used so far. It is calculated, not run: *Python's time ÷ that version's time
  for one photo*. It grows while Python works. When the longest bar reaches the end, the
  scale grows (*full bar = N photos*) and all bars shrink together.

Run the cell and watch the bars (about 15 seconds including the preparation).

In [ ]:
race = notebook_race()

✏️ **Q10.** How long did Python take for the photo in the race, and how does it compare
with its time in Part 5? According to panel 2, how many photos could each version process
in that time? Is the order of the versions the same as in your table from Part 5?

**Your answer:** *(double-click this cell and write it here)*

---
## Part 9. Conclusions (200 words maximum)

Write your conclusions **in your own words** inside the quotes below, then run the cell to
count the words. Answer these points, using your numbers:

1. Where was the biggest jump in speed, and what causes it?
2. Is writing assembly by hand always faster than C? Use ASM v1 vs C v2 as evidence.
3. What did the pointers (v2), the MAC instruction (v3) and NEON (v4) each add?
4. Why does it matter to understand what code does at the lowest level?

In [ ]:
conclusions = """
Write your conclusions here.
"""

words = len(conclusions.split())
print(f"{words} words:", "OK" if words <= 200 else "too long, the maximum is 200")

---
## Hand in

1. Check that every cell has its output and every ✏️ question has your answer.
2. Save (**Ctrl + S**).
3. Export: **File → Save and Export Notebook As… → HTML** (in the classic interface:
   **File → Download as → HTML**).
4. Upload the HTML file to Moodle.